# Individual-Level Spatial Prediction: Data Preparation

While our meta-analytic clustering revealed a macroscopic functional topography of the VMPFC, we must validate whether this generalized, group-level architecture can reliably predict functional organization at the individual subject level.

This notebook sets up a robust, cross-validated machine learning framework (5-fold) using peak activation coordinates derived from individual subjects. Specifically, we aim to precompute the necessary spatial data, Laplacian smoothing matrices, and randomized labels required to perform rigorous permutation testing (2,000 iterations) for evaluating the statistical significance of our subject-level spatial predictions.


`data/group/HCP_S1200_997_tfMRI_ALLTASKS_level2_cohensd_hp200_s4_MSMAll.dscalar.nii` is the publicly released HCP S1200 group-average activation map. Download from HCP ConnectomeDB after accepting the HCP Open Access Data Use Terms; place it under data/group/.

In [ ]:
from pathlib import Path
import numpy as np
from time import time
import scipy.io as sio
import numpy as np
from time import time
from tqdm import trange
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
from sklearn.utils import shuffle
from joblib import Parallel, delayed
import joblib
from threadpoolctl import threadpool_limits
from tqdm_joblib import tqdm_joblib
from scipy.sparse.linalg import splu
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
import numpy as np
from scipy.sparse import eye as sparse_eye


DATA_PATH = Path('../data')
RESULT_PATH = Path('../results')
BRAIN_HEMISPHERE_LIST = ['left', 'right']

LAMBDA_REG = 1e-1
RADIUS = 2.5
PERMUTATION_NUM = 2000
MARGIN = 0.7
SUBJECT_NUM = 677

In [ ]:
def calculate_Sn_new(data, mni_loc, r):
    # Extract three sets of coordinates from the data
    data_1, data_2, data_3 = data[:, 0:3], data[:, 3:6], data[:, 6:9]
    # Precompute the square of the radius to avoid redundant calculations
    r_square = r ** 2

    # Compute distances for each mni_loc voxel using broadcasting
    distances_1 = np.sum((data_1[:, np.newaxis, :] - mni_loc[np.newaxis, :, :]) ** 2, axis=2)
    distances_2 = np.sum((data_2[:, np.newaxis, :] - mni_loc[np.newaxis, :, :]) ** 2, axis=2)
    distances_3 = np.sum((data_3[:, np.newaxis, :] - mni_loc[np.newaxis, :, :]) ** 2, axis=2)

    # Count the number of votes for each mni_loc point
    votes_1 = np.sum(distances_1 < r_square, axis=0)
    votes_2 = np.sum(distances_2 < r_square, axis=0)
    votes_3 = np.sum(distances_3 < r_square, axis=0)

    # Calculate the total number of votes
    votes_all = votes_1 + votes_2 + votes_3

    # Preallocate the Sn matrix
    Sn = np.zeros((mni_loc.shape[0], 3))

    # Set an equal distribution for cases with no votes
    no_votes = (votes_all == 0)
    Sn[no_votes, :] = 1 / 3

    # Calculate proportions for cases where there are votes
    with_votes = ~no_votes
    Sn[with_votes, 0] = votes_1[with_votes] / votes_all[with_votes]
    Sn[with_votes, 1] = votes_2[with_votes] / votes_all[with_votes]
    Sn[with_votes, 2] = votes_3[with_votes] / votes_all[with_votes]

    return Sn


def calculate_laplacian_matrix(mni_loc, margin):
    # 使用广播机制计算成对距离矩阵的平方
    diff = mni_loc[:, np.newaxis, :] - mni_loc[np.newaxis, :, :]
    distance_matrix = np.sqrt(np.sum(diff ** 2, axis=2))
    # A is adjacent matrix
    A = (distance_matrix <= margin).astype(float)
    # set diagal values to 0
    np.fill_diagonal(A, 0)
    # make a sparce matrix
    A_sparse = csr_matrix(A)
    # D is degree matrix
    D = np.diag(A_sparse.sum(axis=1).A.flatten())
    # L is the final laplacian matrix
    L = D - A_sparse

    return L


# Spatial Matrices and Laplacian Smoothing

Individual fMRI peak coordinates are notoriously sparse and noisy. To make this data suitable for machine learning algorithms, we apply a spatial regularization technique.

We map the discrete peak coordinates into our VMPFC target space (separated into left and right hemispheres) and compute a Laplacian matrix (`L`). This mathematical structure allows us to perform heat-kernel-like smoothing, diffusing the discrete peak signals across anatomically continuous voxel boundaries to form robust individual-specific activation topographies.

In [ ]:
raw_data_path = DATA_PATH / 'raw_data.dict.pkl'
if not raw_data_path.exists():

    left_sub_coordinate_file = DATA_PATH / 'individual/left_MPFC_coordinates.mat'
    right_sub_coordinate_file = DATA_PATH / 'individual/right_MPFC_coordinates.mat'
    left_mask_coordinate_file = DATA_PATH / f'individual/rVMPFC_MNI_left.mat'
    right_mask_coordinate_file = DATA_PATH / f'individual/rVMPFC_MNI_right.mat'

    raw_data_dict = dict(
        left_peak=sio.loadmat(left_sub_coordinate_file)['data'],
        right_peak=sio.loadmat(right_sub_coordinate_file)['data'],
        left_MNI=sio.loadmat(left_mask_coordinate_file)['mni_loc'],
        right_MNI=sio.loadmat(right_mask_coordinate_file)['mni_loc'],
    )

    raw_data_dict['left_laplacian_matrix'] = csc_matrix(calculate_laplacian_matrix(raw_data_dict['left_MNI'], MARGIN))
    raw_data_dict['right_laplacian_matrix'] = csc_matrix(calculate_laplacian_matrix(raw_data_dict['right_MNI'], MARGIN))
    joblib.dump(raw_data_dict, raw_data_path, )
else:
    raw_data_dict = joblib.load(raw_data_path)

# Cross-Validation Data Splitting

To ensure our predictive models are highly generalizable and not overfitting to specific individuals, we implement a strict 5-fold cross-validation scheme. We randomly shuffle the subject pool (`SUBJECT_NUM = 677`) and split them into five distinct sets. This specific training and testing index dictionary is saved and fixed to ensure absolute consistency across all subsequent machine learning tasks and random permutations.

In [ ]:
# make index dict for latter permutation dict
train_test_idx_path = DATA_PATH / 'train_test_index.dict.pkl'
all_indices_list = shuffle(np.arange(SUBJECT_NUM), random_state=42)
fold_indices_list = np.array_split(all_indices_list, 5)
if not train_test_idx_path.exists():
    train_test_idx_dict = dict()
    for brain_hemisphere in BRAIN_HEMISPHERE_LIST:
        for fold_i, test_idx in enumerate(fold_indices_list):
            train_idx = np.setdiff1d(all_indices_list, test_idx, assume_unique=True)
            train_test_idx_dict[(brain_hemisphere, fold_i)] = (train_idx, test_idx)
    joblib.dump(train_test_idx_dict, train_test_idx_path)
else:
    train_test_idx_dict = joblib.load(train_test_idx_path)

# Precomputing Null-Distribution Topographies

To statistically validate the accuracy of our prediction algorithm, we need to generate a "null distribution" representing chance performance. We achieve this by permuting the true functional labels of the peak coordinates for each subject.

Because applying Laplacian smoothing (`spsolve`) to generate individual topographies across 5 folds and 2,000 permutations is computationally massive, we define a highly optimized target function (`precompute_fold_data`). This function shuffles the raw labels, recalculates the regularized training signals, extracts the ground-truth test coordinates, and strictly executes in a parallelized environment (`joblib.Parallel`) to dramatically reduce compute time.

In [ ]:
def precompute_fold_data(permutation_i, fold_i, peak_wide_raw, mni_loc, L, I, train_idx, test_idx, r, lambda_reg):
    peak_wide = peak_wide_raw.copy()
    peak_idx_list_raw = [(0, 1, 2), (3, 4, 5), (6, 7, 8)]
    num_subjects = peak_wide.shape[0]

    # 1. shuffle peak labels
    for subject_row_idx in range(num_subjects):
        random_state = subject_row_idx + num_subjects * (fold_i + permutation_i * len(fold_indices_list))
        peak_idx_list = np.array(shuffle(peak_idx_list_raw.copy(), random_state=random_state)).flatten()
        peak_wide[subject_row_idx, :] = peak_wide[subject_row_idx, peak_idx_list]

    # 2. calculate y train
    peak_loc = peak_wide[train_idx, :]
    Sn = calculate_Sn_new(peak_loc, mni_loc, r)
    Sn = csc_matrix(Sn)
    norm_matrix = lambda_reg * spsolve(L + lambda_reg * I, Sn)
    y_train = np.argmax(norm_matrix.toarray(), axis=1) + 1

    # 3. prepare X_test and y_test
    peak_wide_test = peak_wide[test_idx, :]
    X_test = np.vstack([peak_wide_test[:, 0:3], peak_wide_test[:, 3:6], peak_wide_test[:, 6:9]])
    y_test = np.repeat(np.arange(1, 4), peak_wide_test.shape[0])

    # return data
    return {
        'permutation_i': permutation_i, 'fold_i': fold_i,
        'y_train': y_train, 'X_test': X_test, 'y_test': y_test
    }

In [ ]:
for brain_hemisphere in ['left', 'right']:
    permutation_data_path = DATA_PATH / f'individual/{brain_hemisphere}_permutation.list.pkl'
    if permutation_data_path.exists():
        continue

    peak_wide_raw = raw_data_dict[f'{brain_hemisphere}_peak'].copy()
    peak_wide_raw = shuffle(peak_wide_raw, random_state=42)

    mni_loc = raw_data_dict[f'{brain_hemisphere}_MNI']

    L = raw_data_dict[f'{brain_hemisphere}_laplacian_matrix']
    I = sparse_eye(L.shape[0], format='csc')

    print('preparing tasks')
    task_list = []
    for permutation_i in range(PERMUTATION_NUM):
        for fold_i, _ in enumerate(fold_indices_list):
            train_idx, test_idx = train_test_idx_dict[(brain_hemisphere, fold_i)]
            task_list.append((permutation_i, fold_i, train_idx, test_idx))

    result_list = []
    parallel_generator = Parallel(n_jobs=-1, return_as="generator")(
        delayed(precompute_fold_data)(
            p_i, f_i, peak_wide_raw, mni_loc, L, I, tr_idx, te_idx, RADIUS, LAMBDA_REG
        )
        for p_i, f_i, tr_idx, te_idx in task_list
    )
    for res in tqdm(parallel_generator, total=len(task_list), desc=f"Precomputing {brain_hemisphere}"):
        result_list.append(res)
    joblib.dump(result_list, permutation_data_path, compress=3)